# Exp-1 — Gate 1: FDCA vs Dot-Product on LDS

**Phase 1** — train `MlpK` + `Predict` on displaced-predecessor supervision  
**Phase 2** — compare `FDCAScorer` (frozen Phase 1) vs `DotProductScorer` on obs→targets

In [2]:
import sys, os

EXP1_DIR = os.path.abspath('.')          # fdca/exp-1/
FDCA_DIR = os.path.dirname(EXP1_DIR)    # fdca/
for d in [EXP1_DIR, FDCA_DIR]:
    if d not in sys.path:
        sys.path.insert(0, d)

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import wandb

from datasource import LinearDSDataSource
from scorers import FDCAScorer
from model_mlp_k import MlpK
from model_predict import Predict
from train_phase1 import compute_loss, train_one_epoch, train_phase2_one_epoch, Exp1Net

## Config

In [3]:
CFG = dict(
    # LDS
    d_z             = 16,
    obs_dim         = 32,
    seq_len         = 64,
    n_chains        = 8,
    batch_size      = 256,
    seed            = 42,
    # Phase 1
    hidden          = 64,
    p1_epochs       = 30,
    # Phase 2
    d_model         = 64,
    n_heads         = 4,
    p2_epochs       = 30,
    # Optimiser
    steps_per_epoch = 100,
    lr              = 3e-4,
    wd              = 1e-4,
    grad_clip       = 1.0,
    device          = 'cpu',
)

device = torch.device(CFG['device'])
os.makedirs('checkpoints', exist_ok=True)

In [ ]:
run = wandb.init(project='fdca-exp1', name='nb-run', config=CFG)

## Data

In [4]:
ds = LinearDSDataSource(
    d_z=CFG['d_z'], obs_dim=CFG['obs_dim'],
    seq_len=CFG['seq_len'], n_chains=CFG['n_chains'],
    batch_size=CFG['batch_size'], seed=CFG['seed'], device=device,
)

obs_ex, tgt_ex, pidx_ex = ds.sample()
print(f"obs:      {tuple(obs_ex.shape)}")
print(f"targets:  {tuple(tgt_ex.shape)}")
print(f"pred_idx: {tuple(pidx_ex.shape)}  valid={(pidx_ex>=0).sum().item()} / {pidx_ex.numel()}")

obs:      (256, 64, 32)
targets:  (256, 64, 32)
pred_idx: (256, 64)  valid=14336 / 16384


---
## Phase 1 — Train `MlpK` + `Predict`

In [ ]:
mlp_k   = MlpK(CFG['obs_dim'], CFG['d_z'], CFG['hidden']).to(device)
predict = Predict(CFG['d_z']).to(device)
tau     = nn.Parameter(torch.ones(1, device=device))

p1_params = list(mlp_k.parameters()) + list(predict.parameters()) + [tau]
p1_opt    = torch.optim.AdamW(p1_params, lr=CFG['lr'], weight_decay=CFG['wd'])
p1_sched  = torch.optim.lr_scheduler.CosineAnnealingLR(
    p1_opt, T_max=CFG['p1_epochs'] * CFG['steps_per_epoch']
)

p1_log = []

for epoch in range(1, CFG['p1_epochs'] + 1):
    m = train_one_epoch(mlp_k, predict, ds, p1_opt, CFG['grad_clip'], CFG['steps_per_epoch'])
    p1_sched.step()
    p1_log.append(m)
    wandb.log({'phase': 1, 'epoch': epoch,
               'p1/loss': m['loss'], 'p1/mse': m['mse'],
               'p1/z_var': m['z_var'], 'p1/predict_drift': m['predict_drift'],
               'p1/W_id_dist': m['W_id_dist']})

print(f"Phase 1 done  mse={p1_log[-1]['mse']:.3e}  "
      f"drift={p1_log[-1]['predict_drift']:.3f}  W-I={p1_log[-1]['W_id_dist']:.3f}")

### Phase 1 curves

In [ ]:
epochs = list(range(1, len(p1_log) + 1))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Phase 1 — MlpK + Predict training', fontsize=12)

axes[0].semilogy(epochs, [m['mse']          for m in p1_log])
axes[0].set_title('MSE (log)'); axes[0].set_xlabel('epoch'); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [m['predict_drift'] for m in p1_log])
axes[1].set_title('predict drift ||f(z)−z||'); axes[1].set_xlabel('epoch'); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, [m['z_var']         for m in p1_log])
axes[2].set_title('Latent var (collapse guard)'); axes[2].set_xlabel('epoch'); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

### Phase 1 health check

In [ ]:
last = p1_log[-1]
checks = [
    ('MSE near-zero',      last['mse']          < 1e-2,  f"{last['mse']:.2e}"),
    ('No latent collapse', last['z_var']         > 0.01,  f"{last['z_var']:.4f}"),
    ('predict ≠ identity', last['predict_drift'] > 0.05,  f"{last['predict_drift']:.4f}"),
    ('W_predict ≠ I',      last['W_id_dist']     > 0.5,   f"{last['W_id_dist']:.4f}"),
]
print(f"{'Check':<28} {'Pass':>5}  Value")
print('-' * 46)
for name, ok, val in checks:
    print(f"{name:<28} {'✓' if ok else '✗':>5}  {val}")

torch.save(mlp_k.state_dict(),   'checkpoints/phase1_mlp_k.pt')
torch.save(predict.state_dict(), 'checkpoints/phase1_predict.pt')
torch.save(tau,                  'checkpoints/phase1_tau.pt')
print('\nPhase 1 checkpoints saved.')

---
## Phase 2 — FDCA vs Dot-Product

In [ ]:
# ---- 2a: FDCA scorer (frozen Phase 1) ----
scorer_fdca = FDCAScorer(mlp_k, predict, tau)
scorer_fdca.freeze()
frozen = set(scorer_fdca.parameters())

net_fdca   = Exp1Net(CFG['obs_dim'], CFG['d_model'], CFG['n_heads'], scorer=scorer_fdca).to(device)
train_p    = [p for p in net_fdca.parameters() if p not in frozen]
opt_fdca   = torch.optim.AdamW(train_p, lr=CFG['lr'], weight_decay=CFG['wd'])
sched_fdca = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_fdca, T_max=CFG['p2_epochs'] * CFG['steps_per_epoch']
)

# ---- 2b: DotProduct baseline ----
net_dot   = Exp1Net(CFG['obs_dim'], CFG['d_model'], CFG['n_heads'], scorer=None).to(device)
opt_dot   = torch.optim.AdamW(net_dot.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
sched_dot = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_dot, T_max=CFG['p2_epochs'] * CFG['steps_per_epoch']
)

p2_fdca_mse, p2_dot_mse = [], []

for epoch in range(1, CFG['p2_epochs'] + 1):
    m_fdca = train_phase2_one_epoch(
        net_fdca, ds, opt_fdca, CFG['grad_clip'], CFG['steps_per_epoch'], frozen
    )
    m_dot = train_phase2_one_epoch(
        net_dot, ds, opt_dot, CFG['grad_clip'], CFG['steps_per_epoch'], set()
    )
    sched_fdca.step(); sched_dot.step()

    p2_fdca_mse.append(m_fdca)
    p2_dot_mse.append(m_dot)
    wandb.log({'phase': 2, 'epoch': epoch,
               'p2/fdca_mse': m_fdca, 'p2/dotprod_mse': m_dot})

print(f"Final FDCA MSE:    {p2_fdca_mse[-1]:.4e}")
print(f"Final DotProd MSE: {p2_dot_mse[-1]:.4e}")

### Phase 2 comparison curves

In [ ]:
ep = list(range(1, CFG['p2_epochs'] + 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Phase 2 — FDCA vs Dot-Product (LDS displaced-predecessor)', fontsize=12)

for ax, yscale in zip(axes, ['linear', 'log']):
    ax.plot(ep, p2_fdca_mse, label='FDCA (frozen Phase 1)')
    ax.plot(ep, p2_dot_mse,  label='Dot-Product baseline', linestyle='--')
    ax.set_yscale(yscale)
    ax.set_title(f'Prediction MSE ({yscale})')
    ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

### Gate 1 verdict

In [ ]:
f_fdca, f_dot = p2_fdca_mse[-1], p2_dot_mse[-1]
pct = 100 * (f_dot - f_fdca) / f_dot

print(f"Final MSE  FDCA:    {f_fdca:.4e}")
print(f"Final MSE  DotProd: {f_dot:.4e}")
print(f"Improvement:        {pct:+.1f}%")
print()
if f_fdca < f_dot * 0.9:
    verdict = 'PASS — FDCA beats dot-product by >10%. Proceed to Gate 2.'
elif abs(f_fdca - f_dot) < 0.05 * f_dot:
    verdict = 'AMBIGUOUS — check predict_drift; possible identity collapse.'
else:
    verdict = 'FAIL — FDCA does not beat dot-product.'
print(f'Gate 1: {verdict}')

wandb.summary['final_fdca_mse']       = f_fdca
wandb.summary['final_dotprod_mse']    = f_dot
wandb.summary['fdca_improvement_pct'] = pct
wandb.summary['gate1_verdict']        = verdict
wandb.finish()